# Telegraphic SFT Alignment Pipeline (DeepSeek-R1-7B)

## 🎯 What Are We Doing?
We are performing end-to-end 4-bit QLoRA Supervised Fine-Tuning (SFT) on `deepseek-ai/DeepSeek-R1-Distill-Qwen-7B` followed by GSM8K benchmark evaluation, qualitative reasoning trace inspection, and comparative dashboard plotting.

## 💡 Why Are We Doing It?
Standard reasoning models emit verbose, multi-paragraph reasoning traces before giving an answer. This SFT pipeline aligns the model to produce **telegraphic, token-dense `<think>` blocks** (reducing reasoning tokens by ~50%) while maintaining strict `<think>...</think>` format compliance and preserving mathematical accuracy without system prompt regurgitation.

## 🛠️ Code Source & Infrastructure
- **GitHub Repository:** [Hari31416/qwen-grug-finetune](https://github.com/Hari31416/qwen-grug-finetune.git)
- **Frameworks Used:** PyTorch, Hugging Face `transformers`, `peft` (LoRA), `trl` (`SFTTrainer`), `bitsandbytes` (4-bit NF4 quantization).
- **Target Hardware:** 2x NVIDIA T4 GPUs (Kaggle / Google Colab CUDA environment).

## 📊 Data Source
- **Hugging Face Dataset Repository:** [hari31416/qwen-grug-finetune](https://huggingface.co/datasets/hari31416/qwen-grug-finetune) (1,701 stratified reasoning samples across StrategyQA, LogiQA, BoolQ, ANLI, PIQA, and ReClor).
- **Evaluation Benchmark:** [openai/gsm8k](https://huggingface.co/datasets/openai/gsm8k) (Grade School Math reasoning test split).

---

### Notebook Execution Workflow:
1. **Centralized Hyperparameters**: Configure SFT experimental parameters (epochs, batch size, learning rates, limits).
2. **Environment & Data Setup**: Clone repository, install dependencies, and download SFT dataset splits.
3. **Base Model Inference**: Load 4-bit NF4 quantized base model and run sample generation.
4. **SFT Fine-Tuning Execution**: Execute `run_sft_training()` directly in the notebook kernel.
5. **Loss Visualization**: Plot Loss Curves and Learning Rate decay schedule using `plot_latest_training_loss()`.
6. **GSM8K Benchmarking**: Benchmark Base Model vs. Fine-Tuned Model using `run_gsm8k_eval()`.
7. **Qualitative Sample Inspection**: Print side-by-side reasoning traces before and after SFT.
8. **EDA Dashboard**: Plot comparative metrics for Accuracy (%) and Token Count Breakdown.
9. **Export Artifacts Package**: Zip trained adapters, evaluation JSONs, and plot images into a downloadable ZIP file.


## 1. Centralized SFT Hyperparameters & Configuration


In [ ]:
# ==========================================
# ⚙️ SFT EXPERIMENTAL HYPERPARAMETERS & CONFIG
# ==========================================

REPO_URL = "https://github.com/Hari31416/qwen-grug-finetune.git"
REPO_NAME = "qwen-grug-finetune"
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
DATA_DIR = "data"
ADAPTER_OUTPUT_DIR = "adapters"

# Training Hyperparameters
EPOCHS = 1                # 1 epoch for optimal style adaptation without overfitting
TRAIN_BATCH_SIZE = 1      # Per-device batch size (1 max VRAM headroom on T4 GPUs)
GRAD_ACCUM = 8            # Gradient accumulation steps (Effective batch size = 1 * 8 = 8)
LEARNING_RATE = 2e-4      # Peak SFT learning rate
MAX_SEQ_LENGTH = 1536     # Max token sequence length for training
LORA_R = 16               # LoRA rank dimension
LORA_ALPHA = 32           # LoRA alpha scaling factor

# Benchmark Evaluation Hyperparameters
EVAL_LIMIT = None         # Set to None for FULL benchmark evaluation (all 1,000 test samples), or set e.g. 50 for quick debugging
EVAL_BATCH_SIZE = 1       # Per-device evaluation batch size
EVAL_MAX_TOKENS = 1024     # Max generation tokens per GSM8K problem


## 2. Environment Setup & Data Download


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

%pip install -q peft trl bitsandbytes datasets accelerate huggingface_hub matplotlib seaborn pandas pyyaml 'torchao>=0.16.0'


In [ ]:
import sys
if not os.path.exists("scripts") and not os.path.exists(f"{REPO_NAME}/scripts"):
    print(f"Cloning {REPO_URL} into workspace...")
    !git clone {REPO_URL}
    if os.path.exists(REPO_NAME):
        %cd {REPO_NAME}
elif os.path.exists(REPO_NAME) and os.path.exists(f"{REPO_NAME}/scripts"):
    %cd {REPO_NAME}

sys.path.append(".")
from scripts.cuda.cuda_utils import patch_transformers_lazy_imports
from scripts.cuda.download_data import download_hf_data

patch_transformers_lazy_imports()
download_hf_data(output_dir=DATA_DIR)


## 3. Base Model Inference


In [ ]:
import torch
from transformers import BitsAndBytesConfig
from scripts.cuda.cuda_utils import load_causal_lm_model, load_causal_lm_tokenizer
from scripts.cuda.generate_cuda import generate_response

print("Loading Base Model:", MODEL_ID)
is_cuda = torch.cuda.is_available()
model_kwargs = {}
if is_cuda:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs["quantization_config"] = bnb_config
    model_kwargs["device_map"] = "auto"
    model_kwargs["torch_dtype"] = torch.float16

tokenizer = load_causal_lm_tokenizer(MODEL_ID)
model = load_causal_lm_model(MODEL_ID, **model_kwargs)

sample_prompt = "Josh buys a house for $80,000 and puts in $50,000 in repairs. This increased the value of the house by 150%. How much profit did he make?"
print("\n--- Base Model Sample Response ---")
print(generate_response(model, tokenizer, sample_prompt, max_new_tokens=512))


## 4. Execute SFT QLoRA Fine-Tuning


In [ ]:
from scripts.cuda.train_cuda import run_sft_training

print("Starting SFT QLoRA Fine-Tuning...")
sft_trainer = run_sft_training(
    model_arg=MODEL_ID,
    data_dir=DATA_DIR,
    adapter_path=ADAPTER_OUTPUT_DIR,
    epochs=EPOCHS,
    batch_size=TRAIN_BATCH_SIZE,
    grad_accum=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    max_seq_length=MAX_SEQ_LENGTH,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    model=model,
    tokenizer=tokenizer
)


## 5. Plot Training Loss Curves


In [ ]:
from scripts.cuda.plot_loss import plot_latest_training_loss

print("Plotting Training & Validation Loss...")
plot_latest_training_loss()


## 6. GSM8K Benchmark Evaluation (Base vs. SFT)


In [ ]:
import gc
import torch
from scripts.cuda.eval_cuda import run_gsm8k_eval
from peft import PeftModel

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Evaluating Base Model...")
base_summary = run_gsm8k_eval(model, tokenizer, limit=EVAL_LIMIT, batch_size=EVAL_BATCH_SIZE)

latest_adapter = os.path.join(ADAPTER_OUTPUT_DIR, "deepseek-r1-7b/20260804_040058/final_adapters")
if os.path.exists(latest_adapter):
    print("\nEvaluating SFT Fine-Tuned Model...")
    ft_model = PeftModel.from_pretrained(model, latest_adapter)
    if torch.cuda.is_available():
        try:
            ft_model = ft_model.to("cuda")
        except Exception:
            pass
    ft_summary = run_gsm8k_eval(ft_model, tokenizer, limit=EVAL_LIMIT, batch_size=EVAL_BATCH_SIZE, is_adapter=True)


## 7. Qualitative Reasoning Trace Inspection


In [ ]:
import json

b_file = "results/deepseek-r1-7b/baseline/gsm8k.json"
f_file = "results/deepseek-r1-7b/finetuned/gsm8k.json"

if os.path.exists(b_file) and os.path.exists(f_file):
    with open(b_file) as f:
        b_data = json.load(f)["results"]
    with open(f_file) as f:
        f_data = json.load(f)["results"]

    print("=======================================================")
    print("🔍 BEFORE vs AFTER SFT REASONING COMPARISON")
    print("=======================================================")
    for i in range(min(3, len(b_data))):
        b_item, f_item = b_data[i], f_data[i]
        print(f"\n--- Sample {i+1} ---")
        print("Question:", b_item["question"])
        print(f"[BASE] Think: {b_item['thinking_tokens']} tok | Answer: {b_item['answer_tokens']} tok")
        print("Thinking:", b_item["thinking_content"])
        print(f"[SFT] Think: {f_item['thinking_tokens']} tok | Answer: {f_item['answer_tokens']} tok")
        print("Thinking:", f_item["thinking_content"])
        print("Answer:", f_item["answer_content"])
        print("-" * 55)


## 8. Comparative Performance Dashboard


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def get_summary(p):
    if not os.path.exists(p): return None
    with open(p) as f: return json.load(f).get("summary")

b_s = get_summary(b_file)
f_s = get_summary(f_file)

if b_s and f_s:
    plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    cats = ["7B Base", "7B Fine-Tuned"]
    colors = ["#4A90E2", "#50E3C2"]

    # Accuracy
    accs = [b_s["accuracy"] * 100, f_s["accuracy"] * 100]
    bars1 = ax1.bar(cats, accs, color=colors, width=0.45)
    ax1.set_title("GSM8K Accuracy (%)", fontsize=12, fontweight="bold", pad=15)
    ax1.set_ylabel("Accuracy (%)", fontsize=11)
    ax1.set_ylim(0, 100)
    ax1.grid(True, axis="y", linestyle=":", alpha=0.6)
    for bar in bars1:
        h = bar.get_height()
        ax1.annotate(f"{h:.1f}%", xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontweight="bold")

    # Tokens
    think_t = [b_s["mean_thinking_tokens"], f_s["mean_thinking_tokens"]]
    ans_t = [b_s["mean_answer_tokens"], f_s["mean_answer_tokens"]]
    ax2.bar(cats, think_t, label="Thinking Tokens", color="#4A90E2", width=0.45)
    ax2.bar(cats, ans_t, bottom=think_t, label="Answer Tokens", color="#B8E986", width=0.45)
    ax2.set_title("Token Count Breakdown", fontsize=12, fontweight="bold", pad=15)
    ax2.set_ylabel("Average Tokens", fontsize=11)
    ax2.set_ylim(0, 320)
    ax2.legend(loc="upper right")
    ax2.grid(True, axis="y", linestyle=":", alpha=0.6)
    for idx, (t, a) in enumerate(zip(think_t, ans_t)):
        tot = t + a
        ax2.annotate(f"Total: {int(tot)}", xy=(idx, tot), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontweight="bold")

    plt.tight_layout()
    plot_p = os.path.join(ADAPTER_OUTPUT_DIR, "eda_sft_dashboard.png")
    plt.savefig(plot_p, dpi=150, bbox_inches="tight")
    print("Saved dashboard to:", plot_p)
    plt.show()


## 9. Export Artifacts Zip Package


In [ ]:
import zipfile

ZIP_FILE = "kaggle_sft_artifacts.zip"
print(f"Creating downloadable artifacts package: {ZIP_FILE}...")
with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as zipf:
    for target in ["adapters", "results"]:
        if os.path.exists(target):
            for root, dirs, files in os.walk(target):
                for file in files:
                    fp = os.path.join(root, file)
                    zipf.write(fp, os.path.relpath(fp, "."))

if os.path.exists(ZIP_FILE):
    size_mb = os.path.getsize(ZIP_FILE) / (1024 * 1024)
    print(f"\n✅ Packaged artifacts into '{ZIP_FILE}' ({size_mb:.2f} MB)")
